# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIRˆ² dataset using the `mlcroissant` library. This workflow enables programmatic access to the dataset description and records defined by the Croissant schema.

### Dataset Source
FAIRˆ² dataset Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL for FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Pretty-print the dataset metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"License: {meta.license}")
print(f"Version: {meta.version}\n")
print("Keywords:")
pprint.pprint(meta.keywords) 

## 2. Data Overview
Review available record sets, fields, and their IDs. Use the following section to programmatically list available record sets and their properties (each with unique `@id`).

In [ ]:
# List all record sets by @id
print("Available Record Sets and their @id:")
for rs in dataset.metadata.record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description if hasattr(rs, 'description') else ''}")
    if hasattr(rs, 'fields'):
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - Name: {field.name}, @id: {field.id}, dataType: {getattr(field,'data_type',None)}")
    print("")

## 3. Data Extraction
Load data from specific record set(s) into pandas DataFrames for downstream analysis. Reference record set and field `@id` values.

In [ ]:
# Select record sets by their @id (update with actual ids from previous output!)
# Example assumes at least one record set is present; update accordingly.
record_sets = [rs.id for rs in dataset.metadata.record_sets]  # List of record set @ids
print("Selected record sets for extraction:", record_sets)
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nFirst 5 rows from record set {record_set_id}:")
    display(df.head())
    print(f"\nColumns: {df.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping by key attributes. You must refer to columns using their full `@id`.

In [ ]:
# Example: select a record set (replace index if needed)
if len(record_sets) > 0:
    chosen_record_set = record_sets[0]
    df = dataframes[chosen_record_set]

    # Show all available columns
    print("All columns in df:", df.columns.tolist())

    # Pick a numeric field by its @id (update as necessary)
    # Try to autodetect a numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is not None:
        print(f"Numeric field chosen: {numeric_field}")

        threshold = df[numeric_field].median() if len(df[numeric_field]) > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]

        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df[[numeric_field]].head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"First normalized values (z-score):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a non-numeric field if available
        group_field = None
        for col in df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we use `matplotlib` for plotting a numeric field distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets) > 0 and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated end-to-end loading, exploration, filtering, normalization, grouping, and visualization of the FAIRˆ² dataset using `mlcroissant` with all entity references strictly by their `@id`. You may proceed to perform deeper analyses or integrate other machine learning workflows using the extracted DataFrames.